# Validation of Yngve score and Frazier score on CLEAR corpus
### by Veronica Bossio Botero, 03-28-2025

This notebook investigates the sensitivity and robustness of the Yngve score and Frazier score--measures of syntactic complexity--to variations in reading difficulty and text length.

Using the CLEAR Corpus, which includes English-language excerpts tagged with Flesch-Kincaid Grade Level scores, we categorized passages into five readability groups and analyzed whether the Yngve and Frazier scores can accurately reflect increasing text complexity.

After computing the metrics using the OpenWillis pipeline, we conducted:

- **ANOVAs** to assess whether metric distributions vary significantly across grade levels
- **Post-hoc Dunn's tests** to localize where the differences lie
- **Linear regression** to track continuous trends over grade level

To assess dependecy on text length, we "augmented" the CLEAR corpus by sampling substrings of varying lengths for a subset of the excerpts--which yielded CLEAR corpus passages of different lengths--and evaluated whether and how the Yngve and Frazier scores change as a function of sample length. 

In [ ]:
%reload_ext autoreload
%autoreload 2

import openwillis.speech as ows
print(ows.__file__)

import pandas as pd
import os
import numpy as np
import math
from scipy import stats

import random
import nltk
from nltk.tokenize import word_tokenize
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings(action='ignore', category=UserWarning, module='nltk')
warnings.filterwarnings(action='ignore', category=UserWarning, module='tensorflow')

import logging
logging.getLogger('VoskAPI').setLevel(logging.CRITICAL)

# original_stdout = sys.stdout
# sys.stdout = open(os.devnull, 'w')

nltk.download('punkt',quiet=True)
nltk.download('averaged_perceptron_tagger',quiet=True)
nltk.download('wordnet',quiet=True)

# Comment everything below out if you want to validate the openwillis implementation from a fresh environmnt with only Benepar as an additional dependency
# plotly and stats functions are only needed for the CLEAR validation of the measures and visualization. Use a development environment with these dependencies if you want to run the full validation.

import scikit_posthocs as sp
import statsmodels.api as sm

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'
pio.templates.default = 'simple_white'

import stats_visualizations as sv

### Load Augmented CLEAR DataFrame

We load a pre-computed version of the CLEAR corpus that has been **augmented with truncated variants** of the original excerpts.

These augmented excerpts were generated by sampling random-length substrings from each original passage to simulate varying text lengths. For each original excerpt, multiple truncated versions were created with lengths uniformly drawn from a specified range (e.g., 5 to 200 words). This allows us to analyze how metric values vary as a function of excerpt length, independent of content.

The augmentation was performed using a standalone script (`augment_CLEAR_w_diff_lengths.py`) and saved as a CSV for efficient reuse in this and other validation notebooks.

In [ ]:
PATH_TO_CLEAR = '/Users/veronicabossio/Library/Mobile Documents/com~apple~CloudDocs/brooklyn_health/data/CLEAR/augmented_CLEAR.csv'

### Helper Functions for POS Tagging and OpenWillis JSON Formatting

In [ ]:
def get_word_type(tag):
    if tag.startswith('J'):
        return 'Adjective'
    elif tag.startswith('V'):
        return 'Verb'
    elif tag.startswith('N'):
        return 'Noun'
    elif tag.startswith('R'):
        return 'Adverb'
    else:
        return 'Other'
    
def transcript_to_json(transcript):
    """
    convert a transcript string to openwillis-compatible JSON-like format
    by simulating the timing of each word
    """
    word_list = word_tokenize(transcript)

    pos_tagged = nltk.pos_tag(word_list)

    json_data = []
    start_time = 0.0
    for i, (word, tag) in enumerate(pos_tagged):
        end_time = round(start_time + random.uniform(0.2, 0.4), 2)
        json_data.append({
            'conf': 1.0,
            'end': end_time,
            'start': start_time,
            'word': word,
            'old_idx': i,
            'tag': get_word_type(tag)
        })
        start_time = end_time

    return json_data

### Function to compute your measure

In [ ]:
def compute_syntactic_complexity(df):
    """
    Compute syntactic complexity measures for each excerpt in the DataFrame.
    """    

    df['json_output'] = df['Excerpt'].apply(lambda x: transcript_to_json(x))
    df = df.reset_index(drop=True)
    
    yngve_vals = []
    frazier_vals = []
    
    for json_obj in tqdm(df['json_output']):
        aug_summ_df = ows.speech_characteristics(json_obj, option='simple')[2]
        yngve_vals.append(aug_summ_df['yngve_score'].values[0])
        frazier_vals.append(aug_summ_df['frazier_score'].values[0])

    # Assign all at once
    aug_new_df = df.copy()
    aug_new_df['yngve_score'] = yngve_vals
    aug_new_df['frazier_score'] = frazier_vals

    return aug_new_df


In [ ]:
df = pd.read_csv(PATH_TO_CLEAR)

In [ ]:
og_df = df[df['Truncated']==False].copy()

In [ ]:
df_w_measures = compute_syntactic_complexity(og_df)

In [ ]:
df_w_measures.to_pickle('/Users/veronicabossio/Library/Mobile Documents/com~apple~CloudDocs/brooklyn_health/data/CLEAR/yngve_short_clear_openwillis_clean_env.pkl')

## Measures vs. ease of readability and grade level

In [ ]:
ordinal_map = {'Kindergarten': 0, 'Elementary': 1, 'Teen': 2, 'College': 3, 'Adult': 4}
df_w_measures['Grade_Ordinal'] = df_w_measures['Grade Level'].map(ordinal_map)

metrics = ['yngve_score', 'frazier_score']

In [ ]:
# get the original exceprts before the length augmentation (full length)
og_df = df_w_measures[df_w_measures['Truncated'] == False]
grouped = og_df.groupby('Grade Level')

### Continuous grade level

In [ ]:
sv.plot_regression_subplots(metrics, og_df, variable='Flesch-Kincaid-Grade-Level', cols=2)


Both Yngve and Frazier scores show strong, statistically significant positive correlations with Flesch-Kincaid Grade Level. Higher-grade texts tend to exhibit deeper and more complex syntactic structures, as reflected in higher scores.

Frazier shows slightly stronger predictive power, but both measures are robust indicators of structural load in language.


### Binned grade level

In [ ]:

grades = ['Kindergarten', 'Elementary', 'Teen', 'College', 'Adult']

# Plotting the regression subplots for your measures
fig = sv.plot_regression_subplots(metrics, og_df, variable='Grade_Ordinal', cols=3)

fig.update_xaxes(
    tickvals=[0, 1, 2, 3, 4],
    ticktext=grades,
    title=''
)
fig

Yngve and Frazier scores clearly increase with grade level group, showing strong effect sizes and highly significant trends.  
This suggests that syntactic complexity scales systematically with educational difficulty, even when treated as a categorical variable.

## ANOVA for grade level group

In [ ]:
grouped = og_df.groupby('Grade Level')
# Collect results
anova_results = {}

for metric in metrics:
    # Extract metric values for each grade level in order
    groups = [grouped.get_group(level)[metric].dropna() for level in grades] # list of series

    # Run one-way ANOVA and round
    stat, pval = np.round(stats.f_oneway(*groups), 2)
    
    # Store results
    anova_results[metric] = {'F-statistic': stat, 'p-value': pval}

In [ ]:
def make_box_trace(df, metric, grade_order, stat=None, pval=None):
    """
    Returns a list of go.Box traces for each grade level.
    """
    traces = []
    for grade in grade_order:
        traces.append(go.Box(
            y=df[df['Grade Level'] == grade][metric],
            name=grade,
            boxpoints='outliers',
            marker=dict(opacity=0.5, color='royalblue'),
            line=dict(width=1),
            showlegend=False
        ))
    return traces

In [ ]:
cols = 2
rows = math.ceil(len(metrics) / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows, cols=cols,
    subplot_titles=[
        f"{metric.replace('_', ' ').title()}<br>F={anova_results[metric]['F-statistic']:.2f}, "
        f"p={anova_results[metric]['p-value']:.3f}" for metric in metrics
    ]
)

for i, metric in enumerate(metrics):
    row = (i // cols) + 1
    col = (i % cols) + 1
    
    traces = make_box_trace(og_df, metric, grades, 
                            stat=anova_results[metric]['F-statistic'], 
                            pval=anova_results[metric]['p-value'])
    
    for trace in traces:
        fig.add_trace(trace, row=row, col=col)
    
    fig.update_yaxes(title_text=metric.replace('_', ' ').title(), row=row, col=col)
    #fig.update_xaxes(title_text="Grade Level", row=row, col=col)

    fig.update_layout(
    height=rows * 400,
    width=cols * 450,
    title_text="Boxplots of Metrics by Grade Level with ANOVA Results",
    showlegend=False,
    template='plotly_white'
)

fig.show()

Yngve and Frazier scores vary significantly across grade levels (ANOVA p < 0.001), with consistent increases from Kindergarten through Adult.  
This confirms that both measures are sensitive to categorical shifts in linguistic complexity and distinguish between readability groups.

## Pairwise Group Comparisons: Significance and Effect Sizes

To examine where syntactic complexity metrics differ significantly across reading levels, we ran **Dunn’s post-hoc test** following ANOVA. These matrices show:

- **Cohen’s d effect sizes** for each pair of grade groups
- Asterisks (*) indicating statistical significance at p < 0.05

**How to read this:**
- Each cell compares two grade levels (e.g., Kindergarten vs. Adult)
- Colors reflect effect size (Cohen’s d): red = higher value in the row group, blue = higher in the column group
- Asterisks mark significant differences:
  - *p < .05, **p < .01, ***p < .001

**Why this approach?**
- **Dunn’s test** is non-parametric and appropriate for pairwise comparisons after detecting group-level effects
- **Cohen’s d** complements this by quantifying how large the differences are, beyond just significance


In [ ]:
for metric in metrics:
    fig = sv.plot_posthoc_heatmap(og_df, metric)
    fig.show()

Pairwise comparisons reveal large, statistically significant differences in Yngve and Frazier scores between most grade levels (Cohen’s d > 0.8).  
Frazier shows particularly strong separation between Kindergarten and later levels (up to d ≈ 3.9), suggesting it may be especially sensitive to developmental jumps in syntactic complexity.

## Text-sample length dependence

In [ ]:
for metric in metrics:
    fig = sv.plot_length_dependence_by_grade(df_w_measures, metric)
    fig.show()

Yngve and Frazier scores increase modestly with excerpt length, especially in higher-grade texts.  
This suggests some dependence on the amount of input text available, but overall robustness across varying sample lengths.